# BTC Price Forecasting
## Notebook 2 — Preprocessing

This notebook transforms the raw Coinbase CSV into clean, scaled, windowed data
ready to feed directly into the model. Every decision applied here was justified
in Notebook 1.

### Goal

Produce two files:
- `windows.npz` — sliding window arrays ready for `tf.data.Dataset`
- `scaler.pkl` — fitted MinMaxScaler to inverse-transform predictions back to USD

### Steps

1. **Load & reindex** — rebuild the complete 60-second DatetimeIndex to surface 58,354 timestamps missing from the CSV
2. **Forward-fill** — fill all NaN rows in a single pass after reindexing
3. **Trim** — remove everything before 2017-01-01 to eliminate sparse early data
4. **Feature engineering** — log1p volume, cyclical time encodings, drop unused columns
5. **Temporal split** — divide into train / val / test strictly by date, never randomly
6. **Fit & save scaler** — MinMaxScaler fitted on training data only, saved as `scaler.pkl`
7. **Scale** — transform all three splits using the fitted scaler
8. **Build sliding windows** — produce `X (1440, 6)` → `y (1,)` arrays for each split
9. **Save** — write `windows.npz` to Drive


## Setup

Import libraries and mount Google Drive.
All outputs are saved to the same folder as the raw data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import pickle
import os

# ── Paths ──────────────────────────────────────────────────────
DATA_PATH = '/content/drive/MyDrive/TimeSeriesForecasting/coinbase.csv'
OUT_DIR   = '/content/drive/MyDrive/TimeSeriesForecasting/processed'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Windowing config ───────────────────────────────────────────
WINDOW_SIZE = 1440   # 24h lookback at 1-min resolution
HORIZON     = 60     # predict close price 1 hour ahead
STRIDE      = 1      # slide one minute at a time

# ── Time split boundaries ──────────────────────────────────────
TRIM_START = '2017-01-01'
TRAIN_END  = '2018-07-01'
VAL_END    = '2018-10-15'

# ── Feature columns ────────────────────────────────────────────
CLOSE_IDX  = 0       # position of Close in the final feature matrix
FEATURES   = ['Close', 'volume_log', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']

print('Libraries loaded')
print(f'Output directory: {OUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Libraries loaded
Output directory: /content/drive/MyDrive/TimeSeriesForecasting/processed


## Step 1 — Load & reindex

We load the raw CSV and immediately rebuild the complete 60-second DatetimeIndex.
This surfaces the 58,354 timestamps that are absent from the CSV entirely,
making them visible as NaN rows before we fill them in Step 2.

In [ ]:
# Load raw CSV
print('Loading raw data...')
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'  Loaded {len(df):,} rows')

# Parse timestamps and set as index
df['datetime'] = pd.to_datetime(df['Timestamp'], unit='s')
df = df.set_index('datetime').sort_index()

# Rebuild complete 60-second DatetimeIndex
full_index = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq='60s'
)
df = df.reindex(full_index)

print(f'  Rows after reindex: {len(df):,}')
print(f'  New NaN rows surfaced: {df["Close"].isna().sum():,}')
print(f'  Date range: {df.index.min()} → {df.index.max()}')
print('Reindex complete ✓')

Loading raw data...
  Loaded 2,099,760 rows
  Rows after reindex: 2,158,114
  New NaN rows surfaced: 167,423
  Date range: 2014-12-01 05:33:00 → 2019-01-07 22:06:00
Reindex complete ✓


## Step 2 — Forward-fill

All 167,423 NaN rows are now visible. We forward-fill in a single pass —
carrying the last known price forward into every zero-activity minute.
Any rows at the very start of the series with nothing to fill from are dropped.

In [ ]:
print('Forward-filling NaN rows...')
nan_before = df['Close'].isna().sum()

df = df.ffill()

nan_after = df['Close'].isna().sum()
print(f'  NaN rows before : {nan_before:,}')
print(f'  NaN rows after  : {nan_after:,}')

# Drop any leading NaNs at the very start (nothing to forward-fill from)
df = df.dropna(subset=['Close'])
print(f'  Rows after dropna: {len(df):,}')
print('Forward-fill complete ✓')

Forward-filling NaN rows...
  NaN rows before : 167,423
  NaN rows after  : 0
  Rows after dropna: 2,158,114
Forward-fill complete ✓


## Step 3 — Trim to 2017-01-01

We remove everything before 2017-01-01 to eliminate the sparse early period
where gaps of up to 38.4 hours would have introduced long flat-line artifacts
from forward-filling.

In [ ]:
print('Trimming to 2017-01-01...')
rows_before = len(df)

df = df[df.index >= TRIM_START]

rows_after = len(df)
print(f'  Rows before trim : {rows_before:,}')
print(f'  Rows after trim  : {rows_after:,}')
print(f'  Rows removed     : {rows_before - rows_after:,}')
print(f'  Date range       : {df.index.min()} → {df.index.max()}')
print('Trim complete ✓')

Trimming to 2017-01-01...
  Rows before trim : 2,158,114
  Rows after trim  : 1,061,167
  Rows removed     : 1,096,947
  Date range       : 2017-01-01 00:00:00 → 2019-01-07 22:06:00
Trim complete ✓


## Step 4 — Feature engineering

We build the final 6-column feature matrix from the decisions made in Notebook 1.
- `Close` — kept as-is, will be scaled in Step 6
- `Volume_(BTC)` — log1p transformed to compress right skew before scaling
- `hour_sin`, `hour_cos` — cyclical encoding of hour-of-day (period = 24h)
- `dow_sin`, `dow_cos` — cyclical encoding of day-of-week (period = 7 days)

All other columns are dropped.

In [ ]:
print('Engineering features...')

# log1p volume
df['volume_log'] = np.log1p(df['Volume_(BTC)'])

# Cyclical hour-of-day encoding
hour = df.index.hour + df.index.minute / 60.0
df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * hour / 24)

# Cyclical day-of-week encoding
dow = df.index.dayofweek
df['dow_sin'] = np.sin(2 * np.pi * dow / 7)
df['dow_cos'] = np.cos(2 * np.pi * dow / 7)

# Keep only final feature columns
df = df[FEATURES]

print(f'  Feature columns  : {FEATURES}')
print(f'  Shape            : {df.shape}')
print()
print(df.head(3))
print('Feature engineering complete ✓')

Engineering features...
  Feature columns  : ['Close', 'volume_log', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
  Shape            : (1061167, 6)

                      Close  volume_log  hour_sin  hour_cos   dow_sin  dow_cos
2017-01-01 00:00:00  973.35    1.138489  0.000000  1.000000 -0.781831  0.62349
2017-01-01 00:01:00  973.35    1.138489  0.004363  0.999990 -0.781831  0.62349
2017-01-01 00:02:00  973.35    1.138489  0.008727  0.999962 -0.781831  0.62349
Feature engineering complete ✓


## Step 5 — Temporal split

We divide the dataset into train, validation and test sets strictly by date.
No shuffling — future data must never appear in the training set.

In [ ]:
print('Splitting temporally...')

train_df = df[df.index <  TRAIN_END]
val_df   = df[(df.index >= TRAIN_END) & (df.index < VAL_END)]
test_df  = df[df.index >= VAL_END]

print(f'  Train : {len(train_df):,} rows  ({train_df.index.min().date()} → {train_df.index.max().date()})')
print(f'  Val   : {len(val_df):,} rows  ({val_df.index.min().date()} → {val_df.index.max().date()})')
print(f'  Test  : {len(test_df):,} rows  ({test_df.index.min().date()} → {test_df.index.max().date()})')
print()
print(f'  Train : {len(train_df)/len(df)*100:.1f}%')
print(f'  Val   : {len(val_df)/len(df)*100:.1f}%')
print(f'  Test  : {len(test_df)/len(df)*100:.1f}%')
print('Split complete ✓')

Splitting temporally...
  Train : 786,240 rows  (2017-01-01 → 2018-06-30)
  Val   : 152,640 rows  (2018-07-01 → 2018-10-14)
  Test  : 122,287 rows  (2018-10-15 → 2019-01-07)

  Train : 74.1%
  Val   : 14.4%
  Test  : 11.5%
Split complete ✓


## Step 6 — Fit & save scaler

We fit the MinMaxScaler exclusively on the training split.
Fitting on the full dataset would leak val and test price ranges into training —
the model would indirectly know future price levels during training.

The fitted scaler is saved to disk so we can inverse-transform predictions
back to USD in the model and dashboard notebooks.

In [ ]:
print('Fitting scaler on training data only...')

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_df.values)
val_scaled   = scaler.transform(val_df.values)
test_scaled  = scaler.transform(test_df.values)

# Save scaler
scaler_path = os.path.join(OUT_DIR, 'scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f'  Scaler fitted on  : {len(train_df):,} rows')
print(f'  Train scaled range: [{train_scaled.min():.4f}, {train_scaled.max():.4f}]')
print(f'  Val scaled range  : [{val_scaled.min():.4f}, {val_scaled.max():.4f}]')
print(f'  Test scaled range : [{test_scaled.min():.4f}, {test_scaled.max():.4f}]')
print(f'  Scaler saved → {scaler_path}')
print('Scaling complete ✓')

Fitting scaler on training data only...
  Scaler fitted on  : 786,240 rows
  Train scaled range: [0.0000, 1.0000]
  Val scaled range  : [0.0000, 1.0000]
  Test scaled range : [0.0000, 1.0000]
  Scaler saved → /content/drive/MyDrive/TimeSeriesForecasting/processed/scaler.pkl
Scaling complete ✓


---
## Step 7 — Build sliding windows

We slide a window of 1,440 rows across each split to produce input/target pairs.
Each sample consists of:
- `X` — 1,440 consecutive rows of 6 features (24 hours of lookback)
- `y` — the scaled Close price 60 rows after the window ends (1 hour ahead)

**Why stride=60 and not stride=1?**

With stride=1 every window shifts by one minute, producing ~784,000 windows
for the training split alone. At shape (1440, 6) float32 that is approximately
27GB — Colab runs out of RAM and crashes before saving a single file.

Setting stride=60 produces one window per hour instead of one per minute,
reducing the array to ~450MB total — well within Colab limits.
The model still sees the full price history, just sampled at hourly boundaries.

In Notebook 3 we replace this approach entirely with a `tf.data.Dataset`
pipeline that streams windows directly from disk withou

In [ ]:
# Override stride for memory safety
STRIDE = 60   # one window per hour

def make_windows(data, window_size, horizon, stride, close_idx):
    X, y = [], []
    max_start = len(data) - window_size - horizon + 1
    for start in range(0, max_start, stride):
        end = start + window_size
        X.append(data[start:end])
        y.append(data[end + horizon - 1, close_idx])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

print('Building sliding windows (stride=60)...')
print()

print('  Building train windows...')
X_train, y_train = make_windows(train_scaled, WINDOW_SIZE, HORIZON, STRIDE, CLOSE_IDX)
print(f'  X_train: {X_train.shape}  y_train: {y_train.shape}')

print('  Building val windows...')
X_val, y_val = make_windows(val_scaled, WINDOW_SIZE, HORIZON, STRIDE, CLOSE_IDX)
print(f'  X_val  : {X_val.shape}  y_val  : {y_val.shape}')

print('  Building test windows...')
X_test, y_test = make_windows(test_scaled, WINDOW_SIZE, HORIZON, STRIDE, CLOSE_IDX)
print(f'  X_test : {X_test.shape}  y_test : {y_test.shape}')

total_mb = (X_train.nbytes + X_val.nbytes + X_test.nbytes) / 1e6
print()
print(f'  Total memory used: {total_mb:.1f} MB')
print('Windows complete ✓')

Building sliding windows (stride=60)...

  Building train windows...
  X_train: (13080, 1440, 6)  y_train: (13080,)
  Building val windows...
  X_val  : (2520, 1440, 6)  y_val  : (2520,)
  Building test windows...
  X_test : (2014, 1440, 6)  y_test : (2014,)

  Total memory used: 608.7 MB
Windows complete ✓


## Step 8 — Save to disk

We save the windowed arrays and scaler to Drive.
These two files are everything Notebook 3 needs — no raw data required from this point on.

In [ ]:
print('Saving to disk...')

# ── Scaled arrays (for tf.data streaming in Notebook 3) ───────
train_scaled_path = os.path.join(OUT_DIR, 'train_scaled.npy')
val_scaled_path   = os.path.join(OUT_DIR, 'val_scaled.npy')
test_scaled_path  = os.path.join(OUT_DIR, 'test_scaled.npy')

np.save(train_scaled_path, train_scaled)
np.save(val_scaled_path,   val_scaled)
np.save(test_scaled_path,  test_scaled)

print(f'  train_scaled.npy saved → {train_scaled_path}')
print(f'  val_scaled.npy   saved → {val_scaled_path}')
print(f'  test_scaled.npy  saved → {test_scaled_path}')
print()

# ── Windowed arrays (stride=60, for reference) ────────────────
windows_path = os.path.join(OUT_DIR, 'windows.npz')
np.savez_compressed(
    windows_path,
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    X_test=X_test,   y_test=y_test
)
print(f'  windows.npz      saved → {windows_path}')
print(f'  scaler.pkl       saved → {scaler_path}')

print()
print('=' * 50)
print('Preprocessing complete.')
print(f'  train_scaled : {train_scaled.shape}')
print(f'  val_scaled   : {val_scaled.shape}')
print(f'  test_scaled  : {test_scaled.shape}')
print(f'  X_train      : {X_train.shape}')
print(f'  Features     : {FEATURES}')
print(f'  Window       : {WINDOW_SIZE} min lookback → {HORIZON} min ahead')
print('=' * 50)
print()
print('Next step → Notebook 3: Model training')

Saving to disk...
  train_scaled.npy saved → /content/drive/MyDrive/TimeSeriesForecasting/processed/train_scaled.npy
  val_scaled.npy   saved → /content/drive/MyDrive/TimeSeriesForecasting/processed/val_scaled.npy
  test_scaled.npy  saved → /content/drive/MyDrive/TimeSeriesForecasting/processed/test_scaled.npy

  windows.npz      saved → /content/drive/MyDrive/TimeSeriesForecasting/processed/windows.npz
  scaler.pkl       saved → /content/drive/MyDrive/TimeSeriesForecasting/processed/scaler.pkl

Preprocessing complete.
  train_scaled : (786240, 6)
  val_scaled   : (152640, 6)
  test_scaled  : (122287, 6)
  X_train      : (13080, 1440, 6)
  Features     : ['Close', 'volume_log', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
  Window       : 1440 min lookback → 60 min ahead

Next step → Notebook 3: Model training


## Summary

Notebooks 1 and 2 are complete. Here is what we achieved.

### Notebook 1 — Exploration
We audited 2.1M rows of raw Coinbase tick data and made four documented decisions:
how to handle missing data, which features to keep, how to scale, and where to split.
Every choice is justified by the data, not assumed.


### Notebook 2 — Preprocessing
We applied those decisions in a clean sequential pipeline.

**Missing data**
- Reconstructed the full 60-second time grid, surfacing 58,354 hidden missing rows
- Forward-filled all 167,423 NaN rows in a single pass
- Trimmed everything before 2017-01-01, removing 1,096,947 rows of sparse early data

**Feature engineering**
- Kept `Close` as the prediction target and primary input
- Applied log1p transform to `Volume_(BTC)` to compress right skew
- Encoded hour-of-day and day-of-week as cyclical sin/cos features
- Dropped `Open`, `High`, `Low`, `Volume_(Currency)`, `Weighted_Price`

**Splitting and scaling**
- Split chronologically into train (74%), val (14%), test (11%)
- Fitted MinMaxScaler on training data only to prevent leakage
- Saved fitted scaler as `scaler.pkl`

**Windowing**
- Built sliding windows of 1,440 rows lookback → 1 target 60 steps ahead
- Stride of 60 minutes to stay within Colab memory limits
- Produced 13,080 train / 2,520 val / 2,014 test samples

### Output files
| File | Description |
|------|-------------|
| `processed/windows.npz` | Windowed arrays ready for model training |
| `processed/scaler.pkl` | Fitted scaler to inverse-transform predictions to USD |

